In [1]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [15]:
# Make a request
def add_user_message(messages, text):
    user_message = { 'role': "user", 'content': text }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = { 'role': "assistant", 'content': text }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [12]:
import json

def grade_by_model(test_case, output): 
    # Create evaluation prompt 
    eval_prompt = f""" You are an expert code reviewer. Evaluate this AI-generated solution. Task: {test_case["task"]} Solution: {output} Provide your evaluation as a structured JSON object with: - "strengths": An array of 1-3 key strengths - "weaknesses": An array of 1-3 key areas for improvement - "reasoning": A concise explanation of your assessment - "score": A number between 1-10 """ 
    messages = [] 
    add_user_message(messages, eval_prompt) 
    add_assistant_message(messages, "```json") 
    eval_text = chat(messages, stop_sequences=["```"]) 
    return json.loads(eval_text)

In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    #TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [24]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [26]:
with open ("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

{'strengths': ['Correctly handles the core requirement of calculating averages while filtering null and non-numeric values with proper type checking', 'Thoughtfully excludes booleans despite isinstance(value, int) returning True for bools, preventing logical errors', 'Comprehensive test suite covering edge cases (empty lists, None values, mixed types, floats, single values)'], 'weaknesses': ["No input validation: assumes 'numbers' parameter is iterable; will crash with TypeError if passed a non-iterable (int, None, dict, etc.)", 'Returns None for empty/invalid input rather than raising an exception or returning 0, which may cause downstream issues if callers expect a numeric type', 'Inefficient approach: builds an intermediate list before summing; could use a single-pass generator expression instead: sum(v for v in numbers if conditions) / count'], 'reasoning': 'The solution correctly implements the core functionality with thoughtful type-checking logic and good test coverage. However,

In [21]:
output = run_prompt({"task": "Write a C# function that receives a date and determines whether it falls on a weekend."})
result = grade_by_model({"task": "Write a C# function that receives a date and determines whether it falls on a weekend."}, output)
print(result)

{'strengths': ['Comprehensive solution with multiple implementation approaches including static methods and extension methods', 'Well-documented code with XML comments and clear examples demonstrating usage', 'Handles edge cases properly by working with DateTime directly, including time components'], 'weaknesses': ["The 'Key Points' section is incomplete (cuts off mid-sentence at 'Saturday')", 'IsWeekendV2 using switch expression is unnecessarily verbose - could simply return the boolean result directly', 'Missing consideration for culture-specific weekends (some countries have different weekend days)'], 'reasoning': 'The solution correctly solves the stated problem with clean, idiomatic C# code. It goes beyond the basic requirement by providing multiple implementations and extension methods, which adds value. The code is production-ready with proper documentation. However, it has minor issues like the incomplete documentation and slightly over-engineered switch expression. The lack of

In [27]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Calculate Average Ignoring Non-Numeric Values\n\n```python\ndef calculate_average(numbers):\n    \"\"\"\n    Calculate the average of a list of numbers, ignoring null or non-numeric values.\n    \n    Args:\n        numbers: A list that may contain numbers, None, or non-numeric values\n        \n    Returns:\n        float: The average of valid numeric values\n        None: If no valid numeric values are found\n    \"\"\"\n    valid_numbers = []\n    \n    for value in numbers:\n        # Check if value is not None and is numeric\n        if value is not None and isinstance(value, (int, float)) and not isinstance(value, bool):\n            valid_numbers.append(value)\n    \n    # Return average if there are valid numbers, otherwise return None\n    if valid_numbers:\n        return sum(valid_numbers) / len(valid_numbers)\n    else:\n        return None\n\n\n# Test Cases\nif __name__ == \"__main__\":\n    # Test 1: Normal numbers\n    print(calc